In [ ]:
# !pip install yfinance

  Using cached multitasking-0.0.12-py3-none-any.whl
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached curl_cffi-0.13.0-cp39-abi3-win_amd64.whl.metadata (13 kB)
Using cached curl_cffi-0.13.0-cp39-abi3-win_amd64.whl (1.6 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)

   ---------------------------------------- 0/8 [pytz]
   ---------------------------------------- 0/8 [pytz]
   ----- ---------------------------------- 1/8 [peewee]
   ----- ---------------------------------- 1/8 [peewee]
   ----- ---------------------------------- 1/8 [peewee]
   --------------- ------------------------ 3/8 [websockets]
   --------------- ------------------------ 3/8 [websockets]
   --------------- ------------------------ 3/8 [websockets]
   --------------- ------------------------ 3/8 [websockets]
   --------------- ------------------------ 3/8 [websockets]
   -------------------- ------------------- 4/8 [protobuf]
   -------------------- ------------------- 4/

In [ ]:
#!pip install pyyaml

In [2]:
import yaml

config_data = {
    'basket': ['TSLA', 'AAPL', 'MSFT', 'NVDA'],
    'parameters': {
        'start_date': '2023-01-01',
        'end_date': '2024-01-01'
    }
}

with open('basket.yml', 'w') as file:
    yaml.dump(config_data, file, default_flow_style=False)

print("basket.yml generated successfully.")

basket.yml generated successfully.


In [3]:
import json
import yfinance as yf
import pandas as pd
import numpy as np

# 1. YAML Ingestion
with open('basket.yml', 'r') as file:
    config = yaml.safe_load(file)

tickers = config['basket']
start_date = config['parameters']['start_date']
end_date = config['parameters']['end_date']

queries = []
candidates = []

# 2. Iterative Extraction and Processing
for ticker in tickers:
    print(f"Processing sequence for {ticker}...")
    df = yf.download(ticker, start=start_date, end=end_date, progress=False)
    
    if df.empty:
        print(f"Bypassing {ticker}: Empty dataframe.")
        continue

    # Flatten MultiIndex if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
        
    df = df.reset_index()
    
    # Force robust column normalization
    df.columns = [str(col).strip().lower().replace(' ', '_') for col in df.columns]
    
    # Handle index variations
    if 'date' not in df.columns and 'datetime' in df.columns:
        df.rename(columns={'datetime': 'date'}, inplace=True)
        
    df['date'] = df['date'].dt.strftime('%Y-%m-%d')
    
    # Return Calculation
    df['returns'] = df['adj_close'].pct_change() * 100

    def classify_movement(ret):
        if pd.isna(ret):
            return None
        if ret > 0.55:
            return 'rise'
        elif ret < -0.5:
            return 'fall'
        else:
            return 'freeze'

    df['movement'] = df['returns'].apply(classify_movement)
    df = df.dropna().reset_index(drop=True)

    # 3. 5-Day Sliding Window Construction
    for i in range(5, len(df)):
        target_row = df.iloc[i]
        window = df.iloc[i-5:i]
        
        recent_date_list = window['date'].tolist()
        adjusted_close_list = window['adj_close'].tolist()
        
        target_date = target_row['date']
        target_movement = target_row['movement']
        
        queries.append({
            "query_stock": ticker,
            "query_date": target_date,
            "recent_date_list": recent_date_list,
            "adjusted_close_list": adjusted_close_list
        })
        
        candidates.append({
            "candidate_stock": ticker,
            "candidate_date": target_date,
            "candidate_movement": target_movement,
            "recent_date_list": recent_date_list,
            "adjusted_close_list": adjusted_close_list
        })

# 4. JSONL Disk Export
qlist_path = "basket_qlist.json"
clist_path = "basket_clist.json"

with open(qlist_path, 'w') as f:
    for q in queries:
        f.write(json.dumps(q) + '\n')

with open(clist_path, 'w') as f:
    for c in candidates:
        f.write(json.dumps(c) + '\n')

print(f"\nExecution complete.")
print(f"Total Queries Generated: {len(queries)}")
print(f"Total Candidates Generated: {len(candidates)}")

Processing sequence for TSLA...


KeyError: 'adj_close'